In [ ]:
import requests
import re
import sys
import time

# Nhập cookies
zhe_cookie = input("\nEnter ZHE cookie: ").strip()
phpsessid_cookie = input("Enter PHPSESSID cookie: ").strip()

cookie = {
    "ZHE": zhe_cookie,
    "PHPSESSID": phpsessid_cookie
}

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

def grab_archive_sites():
    for i in range(1, 20): # chỉnh số lượng lấy ở đây để lấy nhiều urls hơn
        url = f"https://www.zone-h.org/archive/special={i}"

        print(f"\nĐang lấy: {url}")

        try:
            res = requests.get(
                url,
                cookies=cookie,
                headers=headers,
                timeout=10
            )

        except requests.exceptions.ConnectTimeout:
            print("Timeout khi kết nối. Thử lại sau 5s...")
            time.sleep(5)
            continue

        except requests.exceptions.RequestException as e:
            print(f"Lỗi request: {e}")
            continue

        if res.status_code != 200:
            print(f"Status code lỗi: {res.status_code}")
            continue

        html = res.content

        # Detect CAPTCHA
        if b'captcha' in html.lower():
            print("\n!!! CAPTCHA detected !!!")
            print("1. Mở link này trên trình duyệt:")
            print(url)
            print("2. Giải CAPTCHA")
            print("3. Copy lại cookie mới (ZHE + PHPSESSID)")
            input("\nSau khi xong, nhấn Enter để tiếp tục...")

            # cập nhật cookie mới
            cookie["ZHE"] = input("Nhập lại ZHE cookie: ").strip()
            cookie["PHPSESSID"] = input("Nhập lại PHPSESSID cookie: ").strip()

            continue

        matches = re.findall(b'<td>([^<\n]+)\n\s*</td>', html)

        if not matches:
            print("Không tìm thấy dữ liệu")
            continue

        with open("urls.txt", "a", encoding="utf-8") as f:
            for m in matches:
                try:
                    # domain = m.split(b'/')[0].decode(errors="ignore").strip()
                    full_url = m.decode(errors="ignore").strip()
                    print(f"[+] {full_url}")
                    # f.write(f"http://{domain}\n")
                    f.write(f"http://{full_url}\n")
                except:
                    continue

        time.sleep(2)

def main():
    grab_archive_sites()

if __name__ == "__main__":
    main()